# CleanTest-Agent · Filter 3 模型训练实验记录

- 任务：在 LessIsMore-FSE2025 (469K 行 Methods2Test) 上微调 `Qwen/Qwen2.5-Coder-0.5B`，作为 CleanTest Filter 3 的覆盖率回归器（model mode）。
- 硬件：百度飞桨 AI Studio · NVIDIA A800 80GB。
- 框架：PaddlePaddle 3.0 + PaddleNLP 3.0.0b3 + Python 3.10。

## 流程概览

| 阶段 | 单元格 | 预计时间 |
|---|---|---|
| 1 环境准备与验证 | Cell 2-5 | 5 min |
| 2 解压代码 + 数据 + 模型权重 | Cell 6-9 | 1-3 min |
| 3 Smoke run（10% 数据）| Cell 10-12 | 8 min |
| 4 全量训练（100% × 2 epoch）| Cell 13-16 | 60 min |
| 5 评测 + 输出指标 | Cell 17-19 | 5 min |
| 6 实验小结 | Cell 20 | — |

## 与裸 PaddleNLP Trainer 的差异

1. bf16 mixed precision，避开 fp16 GradScaler 在小批回归任务中的反复 backoff。
2. forward 包 sigmoid + MSE，让回归输出落在 [0, 1]，与 `condition_cover_rate` 的取值范围对齐。
3. 训练前自动把标签 clip 到 [0, 1]（原始 csv 中有 305 行 > 1 的越界值）。
4. 优先走 ModelScope 下载基模权重，hf-mirror 作为 fallback。
5. `BASE_MODEL` 支持本地路径，避免 `snapshot_download` 在第二次跑时仍要求 namespace/repo 形式。

## 飞桨环境提示

本 notebook 默认运行在 conda env `pure-paddle` 下。Cell 4 会一次性把 paddlenlp 3.0、aistudio_sdk 0.2.5、jieba 等装齐；后续所有 `%%bash` 单元格会显式指定 PATH，确保子进程也走 pure-paddle 解释器。

---
## 1 环境准备与验证

In [ ]:
# Cell 3 [Python] 清掉 external-libraries 残留路径，验证 paddle / GPU

import sys
# 飞桨某些镜像会通过 .pth 把 /home/aistudio/external-libraries 推到 sys.path 前面，
# 导致 import 优先命中老版 paddlenlp。这里在每次 kernel 启动时主动剔除。
sys.path = [p for p in sys.path if "external-libraries" not in p]

import paddle
print("Python      :", sys.version.split()[0])
print("sys.executable:", sys.executable)
print("paddle      :", paddle.__version__)
print("CUDA ok     :", paddle.device.is_compiled_with_cuda())
if paddle.device.is_compiled_with_cuda():
    name  = paddle.device.cuda.get_device_name(0)
    props = paddle.device.cuda.get_device_properties(0)
    print("Device      :", name)
    print("Memory GB   :", round(props.total_memory / 1024**3, 1))
    bf16_capable = any(k in name.lower() for k in ["a800", "a100", "h100", "h800"])
    print("bf16 capable:", bf16_capable)

In [ ]:
# Cell 4 [Python] 一次性安装 paddlenlp 3.0 全套依赖（首次启动 kernel 后跑一次即可）

import sys, subprocess, os

# 防御：如果之前删掉过 cwd，os.getcwd() 在子进程启动时会抛 FileNotFoundError
try:
    os.getcwd()
except FileNotFoundError:
    os.chdir("/home/aistudio")
    print("cwd was stale, reset to:", os.getcwd())

py = sys.executable
env = os.environ.copy()
env["PIP_USER"] = "0"   # 飞桨默认 --user，跟我们的 site-packages 安装冲突

MIRROR = "https://pypi.tuna.tsinghua.edu.cn/simple"

def pip_install(*pkgs):
    subprocess.check_call([py, "-m", "pip", "install", "-U", *pkgs, "-i", MIRROR],
                          env=env, cwd="/home/aistudio")

# 1) paddlenlp 3.0 + aistudio_sdk 兼容版本
pip_install("aistudio_sdk==0.2.5")
pip_install("paddlenlp>=3.0.0b3")

# 2) paddlenlp 3.0 的运行时依赖（最小集）
pip_install(
    "jieba", "seqeval", "datasets",
    "scipy>=1.10", "scikit-learn>=1.3",
    "sentencepiece", "safetensors",
    "huggingface_hub>=0.20", "modelscope>=1.30",
    "pandas>=1.5",
)

print("\nAll deps installed.")
print("NOTE: please Restart Kernel (Menu -> Kernel -> Restart) before continuing,")
print("      then run Cell 5 to verify.")

In [ ]:
# Cell 5 [Python] 重启 Kernel 后重新验证：paddlenlp + aistudio_sdk + Qwen2 import

import sys
sys.path = [p for p in sys.path if "external-libraries" not in p]

import paddle, paddlenlp, aistudio_sdk
print("paddle           :", paddle.__version__)
print("paddlenlp ver    :", paddlenlp.__version__)
print("paddlenlp path   :", paddlenlp.__file__)
print("aistudio_sdk ver :", aistudio_sdk.__version__)

from paddlenlp.transformers import Qwen2ForSequenceClassification, AutoTokenizer
from paddlenlp.trainer import Trainer, TrainingArguments, EarlyStoppingCallback
from paddlenlp.data import DataCollatorWithPadding
print("All paddlenlp components import: OK")

---
## 2 解压代码、数据软链、基模权重

In [ ]:
%%bash
# Cell 7 解压代码包并验证关键改动
export PATH=/opt/conda/envs/pure-paddle/bin:$PATH

cd /home/aistudio/work
if [ ! -f cleantest-agent-v3.zip ]; then
    echo "cleantest-agent-v3.zip not found in /home/aistudio/work/"
    exit 1
fi

unzip -oq cleantest-agent-v3.zip
chmod +x cleantest-agent/scripts/train_qwen_baidu.sh

echo "--- repo layout ---"
ls cleantest-agent/
ls cleantest-agent/scripts/

echo
echo "--- patch checks ---"
grep -n "Wrapped model.forward" cleantest-agent/scripts/train_model.py
grep -n "args.bf16"             cleantest-agent/scripts/train_model.py | head -1
grep -n "clip_labels"            cleantest-agent/scripts/train_model.py | head -1
grep -n "is a local directory"   cleantest-agent/scripts/train_qwen_baidu.sh

In [ ]:
# Cell 8 [Python] 给训练脚本打两个就地补丁：
#   A) train_qwen_baidu.sh -> 顶部注入 PATH，确保子进程走 pure-paddle 解释器
#   B) train_model.py      -> _bounded_forward 必须接 self（PaddleNLP 的 wrap_fwd
#                              装饰器以 value(self, *args, **kwargs) 形式调用 forward）。

import pathlib

# ---------- Patch A ----------
pa = pathlib.Path("/home/aistudio/work/cleantest-agent/scripts/train_qwen_baidu.sh")
sa = pa.read_text()
patch_path = "\nexport PATH=/opt/conda/envs/pure-paddle/bin:$PATH\n"
if "pure-paddle/bin" not in sa:
    sa = sa.replace("set -euo pipefail\n",
                    "set -euo pipefail\n" + patch_path)
    pa.write_text(sa)
    print("Patch A: train_qwen_baidu.sh PATH injected")
else:
    print("Patch A: already applied")

# ---------- Patch B ----------
pb = pathlib.Path("/home/aistudio/work/cleantest-agent/scripts/train_model.py")
sb = pb.read_text()

good_sig  = "def _bounded_forward(self, *fwd_args, **fwd_kwargs):"
good_call = "out = _orig_forward(*fwd_args, **fwd_kwargs)"

old_sig_A = "def _bounded_forward(*fwd_args, **fwd_kwargs):"   # zip 自带版本
old_sig_B = "def _bounded_forward(**fwd_kwargs):"             # 早期错误 patch
old_call  = "out = _orig_forward(**fwd_kwargs)"

changed = []
if good_sig not in sb:
    if old_sig_A in sb:
        sb = sb.replace(old_sig_A, good_sig); changed.append("sig A->good")
    elif old_sig_B in sb:
        sb = sb.replace(old_sig_B, good_sig); changed.append("sig B->good")
if good_call not in sb and old_call in sb:
    sb = sb.replace(old_call, good_call); changed.append("call -> good")

if changed:
    pb.write_text(sb)
    print("Patch B applied:", changed)
else:
    print("Patch B: already at good signature")

# ---------- 验证 ----------
import subprocess
print("\n--- verify ---")
subprocess.run(["grep", "-n", "pure-paddle", str(pa)])
subprocess.run(["grep", "-n", "_bounded_forward\\|_orig_forward(", str(pb)])

In [ ]:
%%bash
# Cell 9 数据集软链 + 基模权重就位（不在则用 ModelScope 下）
export PATH=/opt/conda/envs/pure-paddle/bin:$PATH

# ---- 数据 ----
mkdir -p /home/aistudio/work/cleantest-agent/data
SRC=$(find /home/aistudio/data -name "filter_train.csv" 2>/dev/null | head -1)
if [ -z "${SRC}" ]; then
    echo "filter_train.csv not found under /home/aistudio/data; add the dataset first."
    exit 1
fi
ln -sf "${SRC}" /home/aistudio/work/cleantest-agent/data/filter_train.csv
echo "Data: $(ls -lh /home/aistudio/work/cleantest-agent/data/filter_train.csv)"

# ---- 基模权重 ----
WEIGHT_DIR=/home/aistudio/work/cleantest-agent/.ms_cache/Qwen/Qwen2___5-Coder-0___5B
WEIGHT_FILE=${WEIGHT_DIR}/model.safetensors

if [ -f "${WEIGHT_FILE}" ]; then
    SIZE_MB=$(du -m "${WEIGHT_FILE}" | cut -f1)
    echo "Existing weight: ${WEIGHT_FILE} (${SIZE_MB} MB)"
    if [ "${SIZE_MB}" -lt 900 ]; then
        echo "Size too small, removing and redownloading."
        rm "${WEIGHT_FILE}"
    fi
fi

if [ ! -f "${WEIGHT_FILE}" ]; then
    python - <<'PY'
from modelscope import snapshot_download
p = snapshot_download("Qwen/Qwen2.5-Coder-0.5B",
                      cache_dir="/home/aistudio/work/cleantest-agent/.ms_cache")
print("Downloaded to:", p)
PY
fi

ls -lh "${WEIGHT_FILE}"
echo "Weight ready."

---
## 3 Smoke run（10% 数据，1 epoch，约 5–8 分钟）

用 10% 训练集跑 1 个 epoch，作为正式全量训练前的烟囱测试，目的是：

1. 检查 bf16 + sigmoid wrap 后训练 loss 能否在前 100 步进入合理范围（应 < 0.1）；
2. 检查 ModelScope 下下来的 torch 格式权重能否被 PaddleNLP 正确转换；
3. 检查 eval 阶段 MAE / Pearson 等指标能正常算出；
4. 确认单步速度（A800 bf16 sdpa 路径下经验值约 80–100 samples/s）。

`%%bash` 单元格执行期间没有实时输出，单元格状态停在 `[*]` 表示仍在跑。

In [ ]:
%%bash
# Cell 11 Smoke run
export PATH=/opt/conda/envs/pure-paddle/bin:$PATH

cd /home/aistudio/work/cleantest-agent

SKIP_INSTALL=1 \
PRECISION=bf16 \
TRAIN_SUBSAMPLE=0.1 \
EPOCHS=1 \
BATCH_SIZE=32 \
GRAD_ACCUM=1 \
MAX_LEN=512 \
LR=2e-5 \
BASE_MODEL=/home/aistudio/work/cleantest-agent/.ms_cache/Qwen/Qwen2___5-Coder-0___5B \
MODEL_OUT=experiments/results/coverage_run/qwen_smoke_a800 \
LOG_FILE=experiments/results/coverage_run/smoke_a800.log \
bash scripts/train_qwen_baidu.sh

In [ ]:
# Cell 12 [Python] 读 smoke 训练摘要并打印通过判据；如失败展示日志末尾

import json, os
p = "/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/qwen_smoke_a800/training_metrics.json"
log = "/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/smoke_a800.log"

if not os.path.exists(p):
    print("training_metrics.json not found, smoke run did not finish cleanly.")
    print("\n--- last 60 log lines ---")
    if os.path.exists(log):
        with open(log) as f:
            lines = f.readlines()
        print("".join(lines[-60:]))
else:
    with open(p) as f:
        m = json.load(f)
    print(json.dumps(m, indent=2))
    
    v = m.get("validation_metrics", {})
    # paddlenlp Trainer 把 metrics 加了 eval_ 前缀；兼容两种命名
    def pick(k, default=None):
        return v.get(k, v.get("eval_" + k, default))
    mae      = pick("mae", 999)
    mse      = pick("mse", 999)
    pearson  = pick("pearson_r", 0)
    
    print("\n" + "-" * 60)
    print("Smoke run sanity thresholds")
    print("-" * 60)
    print(f"  MAE        = {mae:.4f}     (threshold < 0.10)")
    print(f"  MSE        = {mse:.4f}     (threshold < 0.02)")
    print(f"  Pearson r  = {pearson:.4f}  (threshold > 0.50)")
    
    passed = mae < 0.10 and mse < 0.02 and pearson > 0.5
    print("\nResult:", "PASS" if passed else "FAIL")
    if not passed:
        print("Investigate before launching the full run in Cell 14.")

---
## 4 全量训练（100% 数据，2 epoch，A800 上约 50–80 分钟）

`%%bash` 单元格无法实时刷新长任务的 stdout，所以全量训练拆成三步：

- Cell 14 用 `nohup` 把训练进程放到后台并立即返回；
- Cell 15 是一段轻量监控脚本，在训练期间可以多次重跑，每次会在输出区追加一份带时间戳的状态快照（进程、GPU、日志尾部 25 行）；
- Cell 16 在训练结束后读取 `training_metrics.json`，打印训练超参与验证集指标。

In [ ]:
%%bash
# Cell 14 启动全量训练（后台）
export PATH=/opt/conda/envs/pure-paddle/bin:$PATH

cd /home/aistudio/work/cleantest-agent

if [ -f experiments/results/coverage_run/train_a800.pid ]; then
    OLD_PID=$(cat experiments/results/coverage_run/train_a800.pid)
    if kill -0 "$OLD_PID" 2>/dev/null; then
        echo "Training already running (PID=$OLD_PID). Run 'kill -9 $OLD_PID' before relaunch."
        exit 1
    fi
fi

mkdir -p experiments/results/coverage_run

nohup env PATH=/opt/conda/envs/pure-paddle/bin:$PATH \
    SKIP_INSTALL=1 \
    PRECISION=bf16 \
    EPOCHS=2 \
    BATCH_SIZE=32 \
    GRAD_ACCUM=1 \
    MAX_LEN=512 \
    LR=2e-5 \
    BASE_MODEL=/home/aistudio/work/cleantest-agent/.ms_cache/Qwen/Qwen2___5-Coder-0___5B \
    MODEL_OUT=experiments/results/coverage_run/qwen_0p5b_a800 \
    LOG_FILE=experiments/results/coverage_run/train_a800.log \
    bash scripts/train_qwen_baidu.sh \
    > experiments/results/coverage_run/nohup_a800.out 2>&1 &

PID=$!
echo "$PID" > experiments/results/coverage_run/train_a800.pid
echo "Training launched in background."
echo "  PID    = $PID"
echo "  Log    = experiments/results/coverage_run/train_a800.log"
echo "  Output = experiments/results/coverage_run/qwen_0p5b_a800/"
echo "Run Cell 15 every few minutes to poll progress."

In [ ]:
%%bash
# Cell 15 监控（每次执行追加一份带时间戳的快照）
PID_FILE=/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/train_a800.pid
LOG=/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/train_a800.log

if [ ! -f "$PID_FILE" ]; then
    echo "Training not started yet (Cell 14 has not been run)."
    exit 0
fi

PID=$(cat "$PID_FILE")
echo "========== $(date '+%Y-%m-%d %H:%M:%S') =========="
if kill -0 "$PID" 2>/dev/null; then
    ps -p "$PID" -o pid,etime,pcpu,pmem,cmd | head -2
    STATUS="RUNNING"
else
    echo "Process $PID has exited."
    STATUS="DONE"
fi

echo
echo "-- GPU --"
nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu \
           --format=csv,noheader 2>/dev/null

echo
echo "-- last 25 log lines --"
tail -n 25 "$LOG" 2>/dev/null

if [ "$STATUS" = "DONE" ]; then
    echo
    echo "Training finished. Run Cell 16 to read final metrics."
fi

In [ ]:
# Cell 16 [Python] 训练完成后读 training_metrics.json 并打印

import json, os, time

p = "/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/qwen_0p5b_a800/training_metrics.json"

while not os.path.exists(p):
    print("Waiting for training to finish ...", time.strftime("%H:%M:%S"))
    time.sleep(60)

with open(p) as f:
    m = json.load(f)
print(json.dumps(m, indent=2))

print("\n-- Training summary --")
print(f"  Base model        : {m['base_model']}")
print(f"  Params            : {m['n_params_million']} M")
print(f"  Train rows        : {m['train_rows']}")
print(f"  Valid rows        : {m['valid_rows']}")
print(f"  Epochs            : {m['epochs']}")
print(f"  Effective batch   : {m['effective_batch_size']}")
print(f"  Max seq len       : {m['max_seq_length']}")
print(f"  Precision         : bf16={m.get('bf16', False)}, fp16={m.get('fp16', False)}")
print(f"  Train runtime sec : {m['train_runtime_sec']}")
print(f"  Train runtime min : {m['train_runtime_sec']/60:.1f}")

print("\n-- Validation metrics --")
for k, v in m["validation_metrics"].items():
    if isinstance(v, float):
        print(f"  {k:25s} : {v:.6f}")
    else:
        print(f"  {k:25s} : {v}")

---
## 5 测试集评测（约 5 分钟）

在 47K held-out 测试集上跑评测，输出 `test_metrics.json`。该文件包含 9 个用于报告 §7.5 `tab:filter3-model-mode` 表格的字段（MAE / MSE / RMSE / R² / Pearson r / Spearman ρ / Precision / Recall / F1），以及 4 项混淆矩阵。

In [ ]:
%%bash
# Cell 18 跑测试集评测
export PATH=/opt/conda/envs/pure-paddle/bin:$PATH

cd /home/aistudio/work/cleantest-agent

python scripts/evaluate_model.py \
    --input_csv  experiments/results/coverage_run/splits/test.csv \
    --model_path experiments/results/coverage_run/qwen_0p5b_a800 \
    --output_predictions experiments/results/coverage_run/test_pred_a800.csv \
    --batch_size 64 \
    --max_seq_length 512 \
    --threshold 0.01

echo
echo "-- test_metrics.json --"
cat experiments/results/coverage_run/test_metrics.json

In [ ]:
# Cell 19 [Python] 打印 9 项指标 + 自动生成报告所需的 LaTeX 表格片段

import json
p = "/home/aistudio/work/cleantest-agent/experiments/results/coverage_run/test_metrics.json"
with open(p) as f:
    m = json.load(f)

print("-" * 60)
print("Filter 3 (Qwen2.5-Coder-0.5B) test set performance")
print("-" * 60)
print(f"  Test set size       : {m['n']:>12,}")
print()
print("  Regression")
print(f"  MAE                 : {m['mae']:.4f}")
print(f"  MSE                 : {m['mse']:.4f}")
print(f"  RMSE                : {m['rmse']:.4f}")
print(f"  R^2                 : {m['r2']:.4f}")
print(f"  Pearson r           : {m['pearson_r']:.4f}")
print(f"  Spearman rho        : {m['spearman_rho']:.4f}")
print()
print(f"  Classification (threshold = {m['threshold']})")
print(f"  Precision           : {m['precision']:.4f}")
print(f"  Recall              : {m['recall']:.4f}")
print(f"  F1                  : {m['f1']:.4f}")
print()
print("  Confusion")
c = m['confusion']
print(f"  TP={c['tp']:>6,}  FP={c['fp']:>6,}  FN={c['fn']:>6,}  TN={c['tn']:>6,}")

print()
print("-" * 60)
print("LaTeX snippet for Table tab:filter3-model-mode")
print("-" * 60)
print(rf"""\begin{{tabular}}{{lr}}
\toprule
\textbf{{Metric}} & \textbf{{Value}} \\
\midrule
\multicolumn{{2}}{{l}}{{\emph{{Regression on continuous coverage}}}} \\
\quad MAE                              & {m['mae']:.4f}        \\
\quad MSE                              & {m['mse']:.4f}        \\
\quad RMSE                             & {m['rmse']:.4f}       \\
\quad $R^2$                            & {m['r2']:.4f}         \\
\quad Pearson $r$                      & {m['pearson_r']:.4f}  \\
\quad Spearman $\rho$                  & {m['spearman_rho']:.4f} \\
\midrule
\multicolumn{{2}}{{l}}{{\emph{{Threshold-aware classification ($\tau = 0.01$)}}}} \\
\quad Precision                        & {m['precision']:.4f}  \\
\quad Recall                           & {m['recall']:.4f}     \\
\quad F1                               & {m['f1']:.4f}         \\
\bottomrule
\end{{tabular}}""")

---
## 6 实验小结

### 待办（在本机完成）

1. 下载产物：
   - `experiments/results/coverage_run/test_metrics.json`
   - `experiments/results/coverage_run/qwen_0p5b_a800/training_metrics.json`
   - `experiments/results/coverage_run/train_a800.log`
2. 在 `report/main.tex` 中：
   - 用 Cell 19 输出的 LaTeX 片段替换 `tab:filter3-model-mode` 占位；
   - `grep -n TBD report/main.tex` 把剩余 9 个 `TBD` 替换为对应字段；
   - 把摘要、§4 FR5、§5.6、§8 Threats、§9 Future Work 中关于 "未训未测 / un-validated" 的措辞改为已训已测的描述；
   - 重新编译 LaTeX。

### 实验记录的对外用途

- 本 notebook 与所引用的 `scripts/` 加上挂载数据集，构成可重跑的实验工件。
- Cell 15 的多次执行结果会作为带时间戳的进度记录保留在 notebook 中，可作为训练耗时与稳定性的参考。
- Cell 19 输出的 LaTeX 片段可直接拷贝进报告，免除手抄数字带来的笔误。

### 各 cell 角色一览

| Cell | 类型 | 作用 |
|---|---|---|
| 1, 2 | Markdown | 文档头与流程概览 |
| 3 | Python | sys.path 净化 + paddle / GPU 自检 |
| 4 | Python | 一次性安装 paddlenlp 3.0 + aistudio_sdk 0.2.5 + 周边依赖 |
| 5 | Python | 重启 Kernel 后重新验证 import 链 |
| 6 | Markdown | 第 2 阶段段落标题 |
| 7 | %%bash | 解压 v3 zip + 4 项 patch 命中 |
| 8 | Python | 注入 PATH + 修复 _bounded_forward 签名 |
| 9 | %%bash | 数据软链 + 基模权重就位 |
| 10 | Markdown | 第 3 阶段段落标题 |
| 11 | %%bash | Smoke run（10% × 1 ep） |
| 12 | Python | smoke 通过判据 / 失败时打印日志 |
| 13 | Markdown | 第 4 阶段段落标题 |
| 14 | %%bash | 启动全量训练（nohup） |
| 15 | %%bash | 进度监控（重复执行） |
| 16 | Python | training_metrics 摘要 |
| 17 | Markdown | 第 5 阶段段落标题 |
| 18 | %%bash | 测试集评测 |
| 19 | Python | 9 指标 + 自动生成 LaTeX |
| 20 | Markdown | 实验小结 |